# 4. Из углового решения к заряду и временному профилю

Здесь используется рабочий решатель, но каждый этап обращения имеет отдельный
смысл и отдельную проверку. Это небольшой учебный расчёт; он не устанавливает
точность во всех геометриях.

In [ ]:
from pathlib import Path
import sys, time, platform
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Notebook can be started from the repo root or notebooks/course.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/lighthit').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start this notebook inside the LightHit repository')
sys.path.insert(0, str(ROOT / 'src'))
from lighthit import Medium, SolverSettings, PointGreenSolver

# Explicit public test medium. No private provider is imported.
medium = Medium(0.04, 0.05, 0.7, 1.35, 450.0, 'course-synthetic')
np.set_printoptions(precision=7, suppress=True)
print('Python:', sys.executable)
print('Platform:', platform.platform())

## 4.1. Угловой интеграл по пространственной частоте

Пусть $\nu=\widehat{\mathbf r}\cdot\mathbf s_0$. Тождество плоской волны даёт
$$\int e^{i\mathbf k\cdot\mathbf r}P_\ell(\widehat{\mathbf k}\cdot\mathbf s_0)d\Omega_k
=4\pi i^\ell j_\ell(kr)P_\ell(\nu).$$
После этого остаются радиальные интегралы, а не трёхмерный FFT:
$$K^{(\ge2)}(r,\nu,\omega)=\sum_{\ell=0}^J P_\ell(\nu)\mathcal R_\ell^{(\ge2)}(r,\omega).$$
$\mathcal R_\ell$ определена в следующей работе о кэше.

Дальше решаем один направленный источник с $\nu=1/2$, $r=20$ м.

In [ ]:
settings=SolverSettings(24,120,4.,.05,8)
omega=np.linspace(0,1.2,241)
t0=time.perf_counter()
result=PointGreenSolver(medium,settings).solve(omega,[np.sqrt(300.),0.,10.],direction=[0,0,1])
print('Elapsed [s]:',time.perf_counter()-t0)
print('Charge [m^-2]:',result.charge_per_m2)
print('Stage timings:',result.timings_s)
fig,ax=plt.subplots();ax.plot(omega,result.components[:,0,:].sum(1).real,label='real');ax.plot(omega,result.components[:,0,:].sum(1).imag,label='imag');ax.set(xlabel='omega [rad/ns]',ylabel='spectrum [m^-2]');ax.legend();plt.show()

## 4.2. Что относится к первому порядку

Рабочий `PointGreenSolver` возвращает
$K^{(0)}+K_{\rm HG}^{(1)}+K_L^{(\ge2)}$.
Первый порядок берётся из координатного интеграла с полной HG-функцией,
а не из усечённого углового ряда. Сохранённые компоненты нужно проверять
до суммирования. Ниже отдельно видны вклад одного рассеяния и остаток.

При направленном источнике на выбранной геометрии $K^{(0)}=0$.

In [ ]:
front=result.front_time_ns[0]
edges=np.arange(front-20,front+401,2.)
raw=result.readout(edges,sigma_ns=0.)
blur=result.readout(edges,sigma_ns=3.)
print('Readout type:',type(raw))

## 4.3. Почему интегрируем по бинам

Для центра бина $t_b$ и ширины $\Delta t$:
$$\int_{t_b-\Delta t/2}^{t_b+\Delta t/2}e^{-i\omega t}dt
=\Delta t\,\operatorname{sinc}(\omega\Delta t/2)e^{-i\omega t_b}.$$
Аппаратное гауссовское размытие добавляет $e^{-\omega^2\sigma^2/2}$.
Заряд — значение спектра при нулевой частоте, не значение импульса в максимуме.

При равномерной сетке $\Delta\omega$ сумма имеет период $2\pi/\Delta\omega$.
Проверять причинность нужно до readout: гауссовское размытие само даёт
сигнал раньше исходного фронта.

In [ ]:
# TimeProfile keeps integrated bin charges, not samples of a density.
print('Array shape (receiver, bin, order):', raw.components.shape)
print('Raw diagnostics:', raw.diagnostics)
print('Readout diagnostics:', blur.diagnostics)
fig, ax = plt.subplots()
ax.plot(raw.centers_ns, raw.rate_per_m2_ns[0], label='No instrument smearing')
ax.plot(blur.centers_ns, blur.rate_per_m2_ns[0], label='Gaussian sigma = 3 ns')
ax.axvline(front, linestyle='--', label='Geometrical front')
ax.set(xlabel='Time [ns]', ylabel='Response [m^-2 ns^-1]')
ax.legend(); plt.show()
fig, ax = plt.subplots()
widths = np.diff(raw.edges_ns)
for order, label in enumerate(['Direct', 'One scattering', 'Two or more']):
    ax.plot(raw.centers_ns, raw.components[0, :, order] / widths, label=label)
ax.set(xlabel='Time [ns]', ylabel='Unsmeared component [m^-2 ns^-1]')
ax.legend(); plt.show()
period = 2*np.pi/(omega[1]-omega[0])
print('Period associated with frequency spacing [ns]:', period)
print('Integrated charge in this finite time window [m^-2]:', raw.values_per_m2.sum())
print('Charge from omega=0, whole time axis [m^-2]:', result.charge_per_m2)

## Задания

Менять по одному $L$, $J$, $k_{\max}$, шаг радиальной квадратуры,
$\omega_{\max}$ и $\Delta\omega$. Вычислять разность заряда и $L^1$-разность
временных бинов, делённую на заряд. Отдельно записывать отрицательную массу
и интеграл до фронта. Не обнулять отрицательные бины ради красивого графика.

Код: `green.py`, `readout.py`, `single.py`. Следующая работа отделяет
подготовку пространства от многократных запросов.